# RoboVision Challenge - DataHack 3.0

**Task.** Each sample is a 640x480 ray-traced tabletop scene (coloured cubes + bowls on a dark
table), a JSON with the exact camera intrinsics/extrinsics, and a 6-step action sequence.
An action is `[x, y, z, grip]`, so 24 numbers have to be predicted per test image.

**What this notebook does**

| Step | Idea |
|---|---|
| 1 | **Free 12 of 24 targets.** `(z, grip)` of every step is a pure function of `task_type` - verified on all 5000 train rows - so it is copied from a lookup table instead of predicted. |
| 2 | **Kill the camera nuisance.** Using the supplied `K` and `[R\|t]` each frame is warped into a canonical top-down (bird's-eye) view of the table plane. Every scene is then aligned, whatever the camera pose was. |
| 3 | **Localise, don't regress.** A small U-Net predicts one heat-map per step in BEV space; a sub-pixel soft-argmax turns it into metres. Heat-map + soft-argmax is far more precise than direct coordinate regression. |
| 4 | Train on **2x T4** with `DataParallel` + AMP, validate on a held-out split, then fine-tune on 100 % of the data and write `submission.csv`. |

Runtime: roughly 25-45 min end-to-end on T4x2.

## 1 - Setup

In [ ]:
import os, sys, json, math, time, glob, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| device", dev, "| #GPUs", NGPU)
for k in range(NGPU):
    p = torch.cuda.get_device_properties(k)
    print(f"  GPU{k}: {p.name}  {p.total_memory/2**30:.1f} GB")

In [ ]:
def find_root():
    '''Locate the Challenge_Dataset folder (Kaggle input or a local checkout).'''
    pats = ["/kaggle/input/*/Challenge_Dataset", "/kaggle/input/Challenge_Dataset",
            "/kaggle/input/*/*/Challenge_Dataset", "./Challenge_Dataset",
            "../Challenge_Dataset", "."]
    for p in pats:
        for c in sorted(glob.glob(p)):
            if (os.path.exists(os.path.join(c, "train.csv"))
                    and os.path.exists(os.path.join(c, "test.csv"))):
                return os.path.abspath(c)
    for c in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        return os.path.dirname(c)
    raise FileNotFoundError("Challenge_Dataset not found")

ROOT = find_root()
print("DATA ROOT:", ROOT)
print(sorted(os.listdir(ROOT))[:10])

train_df = pd.read_csv(os.path.join(ROOT, "train.csv"))
test_df  = pd.read_csv(os.path.join(ROOT, "test.csv"))
print(train_df.shape, test_df.shape)
train_df[["image_id", "task_type"]].head()

## 2 - The 12 "free" targets

Column layout: `action_k = "x,y,z,grip"`. Group the strings by `(task_type, step)` and count the
distinct `(z, grip)` pairs: there is exactly **one** per group. Half of the submitted numbers do
not depend on the image at all, we simply copy them from the training table.

In [ ]:
NSTEP = 6

def parse_actions(df):
    '''-> float array (N, 6, 4)'''
    return np.stack([df[f"action_{i}"].str.split(",", expand=True).astype(float).values
                     for i in range(NSTEP)], 1)

A_tr = parse_actions(train_df)
print("actions:", A_tr.shape)

for t in ["task1", "task2"]:
    m = (train_df.task_type == t).values
    for s in range(NSTEP):
        u = np.unique(np.round(A_tr[m, s, 2:], 4), axis=0)
        assert len(u) == 1, (t, s, u)
print("OK - (z, grip) is unique per (task_type, step)")

ZG = {}                      # (task_type, step) -> (z, grip)
for t in train_df.task_type.unique():
    m = (train_df.task_type == t).values
    for s in range(NSTEP):
        ZG[(t, s)] = tuple(np.round(A_tr[m, s, 2:].mean(0), 4))
print({k: v for k, v in list(ZG.items())[:6]})

## 3 - Camera geometry -> canonical bird's-eye view

The JSON gives `intrinsics` (3x3) and `extrinsics` (4x4, **camera->world**, OpenCV convention:
x right, y down, z forward). Re-building the rotation from `camera_position` / `target_position` /
`up_vector` reproduces it exactly, so the calibration can be trusted.

For every pixel of a canonical top-down grid over the table (`x in [-0.45, 0.45]`,
`y in [-0.90, -0.10]`, z = 0) we project onto the table plane and bilinearly sample the frame.
The result is a 256x288 view where the same object always lands on the same pixel, no matter
where the camera was.

In [ ]:
X0, X1 = -0.45, 0.45
Y0, Y1 = -0.90, -0.10
PX = 320.0
BEV_W = int(round((X1 - X0) * PX))   # 288
BEV_H = int(round((Y1 - Y0) * PX))   # 256

class Cam:
    '''Camera with OpenCV convention; extrinsics = camera->world.'''
    def __init__(self, d):
        self.K = np.array(d["intrinsics"], float)
        self.E = np.array(d["extrinsics"], float)
        self.R = self.E[:3, :3]
        self.t = self.E[:3, 3]
    def project(self, P):
        Xc = (np.asarray(P, float) - self.t) @ self.R
        z = Xc[..., 2]
        uv = Xc[..., :2] / z[..., None]
        return uv @ self.K[:2, :2].T + self.K[:2, 2], z

_mesh_cache = None
def _mesh():
    global _mesh_cache
    if _mesh_cache is None:
        xs = X0 + (np.arange(BEV_W) + 0.5) / PX
        ys = Y0 + (np.arange(BEV_H) + 0.5) / PX
        XX, YY = np.meshgrid(xs, ys)
        _mesh_cache = np.stack([XX, YY, np.zeros_like(XX)], -1).reshape(-1, 3)
    return _mesh_cache

def birdseye(img, cam):
    P = _mesh()
    uv, z = cam.project(P)
    u, v = uv[:, 0], uv[:, 1]
    h, w = img.shape[:2]
    ok = (z > 0.05) & (u >= 0) & (u < w - 1) & (v >= 0) & (v < h - 1)
    u0 = np.clip(u.astype(np.int32), 0, w - 2)
    v0 = np.clip(v.astype(np.int32), 0, h - 2)
    fu = (u - u0).astype(np.float32)[:, None]
    fv = (v - v0).astype(np.float32)[:, None]
    f = img.astype(np.float32)
    c = (f[v0, u0] * (1 - fu) * (1 - fv) + f[v0, u0 + 1] * fu * (1 - fv)
         + f[v0 + 1, u0] * (1 - fu) * fv + f[v0 + 1, u0 + 1] * fu * fv)
    out = np.zeros((BEV_H * BEV_W, 3), np.float32)
    out[ok] = c[ok]
    return out.reshape(BEV_H, BEV_W, 3).astype(np.uint8)

def w2b(x, y):   return (x - X0) * PX, (y - Y0) * PX
def b2w(bx, by): return bx / PX + X0, by / PX + Y0

In [ ]:
# sanity check: BEV of a training frame with that row's 6 targets on top
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

i = 3
img = np.asarray(Image.open(os.path.join(ROOT, train_df.image_path[i].replace("/", os.sep))).convert("RGB"))
cam = Cam(json.load(open(os.path.join(ROOT, train_df.camera_params_path[i].replace("/", os.sep)))))
bev = birdseye(img, cam)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(img); ax[0].set_title("raw frame (random camera pose)"); ax[0].axis("off")
ax[1].imshow(bev)
bx, by = w2b(A_tr[i, :, 0], A_tr[i, :, 1])
ax[1].scatter(bx, by, s=90, facecolors="none", edgecolors="r", linewidths=2)
for s in range(NSTEP):
    ax[1].text(bx[s] + 6, by[s], str(s), color="r", fontsize=11)
ax[1].set_title("canonical BEV + ground-truth targets"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("BEV covers", round(float((bev.sum(-1) > 0).mean()), 3), "of the canvas on this frame")

## 4 - Pre-render the BEV cache

Decoding a PNG costs ~40 ms, so the warp is done once per image (in parallel) and stored as a
`uint8` array. 6000 frames x 256 x 288 x 3 is about 1.3 GB, which fits comfortably in RAM.

In [ ]:
from multiprocessing import Pool

def _bev_one(args):
    root, ipath, cpath = args
    img = np.asarray(Image.open(os.path.join(root, ipath.replace("/", os.sep))).convert("RGB"))
    cam = Cam(json.load(open(os.path.join(root, cpath.replace("/", os.sep)))))
    return birdseye(img, cam)

def build_bev(df, root=ROOT, nproc=4, tag=""):
    out = np.zeros((len(df), BEV_H, BEV_W, 3), np.uint8)
    jobs = [(root, df.image_path[i], df.camera_params_path[i]) for i in range(len(df))]
    t0 = time.time()
    ok = False
    if nproc > 1:
        try:
            with Pool(nproc) as pool:
                for k, arr in enumerate(pool.imap(_bev_one, jobs, chunksize=8)):
                    out[k] = arr
                    if k % 1000 == 0:
                        print(f"  {tag} {k}/{len(df)}  {time.time()-t0:.0f}s", flush=True)
            ok = True
        except Exception as e:
            print(f"{tag}: parallel build unavailable ({type(e).__name__}), using serial")
    if not ok:
        for k, job in enumerate(jobs):
            out[k] = _bev_one(job)
            if k % 500 == 0:
                print(f"  {tag} {k}/{len(df)}  {time.time()-t0:.0f}s", flush=True)
    print(f"{tag} BEV cache ready in {time.time()-t0:.0f}s")
    return out

t0 = time.time()
BEV_TR = build_bev(train_df, nproc=4, tag="train")
BEV_TE = build_bev(test_df,  nproc=4, tag="test")
print("total %.1f min" % ((time.time() - t0) / 60))

## 5 - Model

Small U-Net (4 down-sampling stages, 6-channel heat-map head at stride 2 plus a global
regression head used as an auxiliary loss). `task_type` is injected as an extra constant
channel so a single network serves both task families.

In [ ]:
def conv_bn(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, bias=False), nn.BatchNorm2d(cout), nn.SiLU(True))

class Res(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = conv_bn(c, c)
        self.c2 = nn.Conv2d(c, c, 3, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(c)
    def forward(self, x):
        return F.silu(x + self.bn(self.c2(self.c1(x))), True)

class Net(nn.Module):
    def __init__(self, cin=4, width=32, nstep=NSTEP):
        super().__init__()
        w = width
        self.s1 = nn.Sequential(conv_bn(cin, w, 3, 2), Res(w))
        self.s2 = nn.Sequential(conv_bn(w, w*2, 3, 2), Res(w*2))
        self.s3 = nn.Sequential(conv_bn(w*2, w*4, 3, 2), Res(w*4))
        self.s4 = nn.Sequential(conv_bn(w*4, w*8, 3, 2), Res(w*8))
        self.u3 = conv_bn(w*8 + w*4, w*4)
        self.u2 = conv_bn(w*4 + w*2, w*2)
        self.u1 = conv_bn(w*2 + w, w)
        self.head = nn.Conv2d(w, nstep, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.reg = nn.Sequential(nn.Linear(w*8, 256), nn.SiLU(True), nn.Linear(256, nstep*2))
        nn.init.zeros_(self.head.weight)
        nn.init.constant_(self.head.bias, -2.0)
        self.nstep = nstep
    def forward(self, x):
        f1 = self.s1(x); f2 = self.s2(f1); f3 = self.s3(f2); f4 = self.s4(f3)
        reg = self.reg(self.pool(f4).flatten(1)).view(-1, self.nstep, 2)
        y = F.interpolate(f4, scale_factor=2, mode="nearest"); y = self.u3(torch.cat([y, f3], 1))
        y = F.interpolate(y, scale_factor=2, mode="nearest"); y = self.u2(torch.cat([y, f2], 1))
        y = F.interpolate(y, scale_factor=2, mode="nearest"); y = self.u1(torch.cat([y, f1], 1))
        return self.head(y), reg

print("params %.2f M" % (sum(p.numel() for p in Net().parameters()) / 1e6))

In [ ]:
SIGMA = 6.0     # gaussian radius in BEV pixels (1 px = 3.1 mm)

def heatmap_targets(bxy, h, w, sigma=SIGMA, scale=2.0):
    ys = (torch.arange(h, device=bxy.device, dtype=torch.float32) + .5) * scale
    xs = (torch.arange(w, device=bxy.device, dtype=torch.float32) + .5) * scale
    cx = bxy[..., 0][..., None, None]
    cy = bxy[..., 1][..., None, None]
    d2 = (xs[None, None, None, :] - cx) ** 2 + (ys[None, None, :, None] - cy) ** 2
    return torch.exp(-d2 / (2 * sigma ** 2))

def focal_heatmap_loss(logits, tgt, eps=1e-4):
    p = torch.sigmoid(logits).clamp(eps, 1 - eps)
    npos = (tgt > 0.9).sum().clamp(min=1)
    pos = -((1 - p) ** 2 * torch.log(p) * tgt).sum()
    neg = -((1 - tgt) ** 4 * p ** 2 * torch.log(1 - p)).sum()
    return (pos + neg) / npos

def soft_argmax(hm, scale=2.0, beta=8.0):
    B, S_, h, w = hm.shape
    p = torch.softmax(hm.reshape(B, S_, -1) * beta, -1).reshape(B, S_, h, w)
    ys = (torch.arange(h, device=hm.device, dtype=torch.float32) + .5) * scale
    xs = (torch.arange(w, device=hm.device, dtype=torch.float32) + .5) * scale
    return torch.stack([(p.sum(2) * xs).sum(-1), (p.sum(3) * ys).sum(-1)], -1)

## 6 - Datasets and training loop

In [ ]:
class BEVDataset(torch.utils.data.Dataset):
    def __init__(self, bev, xy, task, augment=False, flip=False):
        self.bev, self.xy, self.task = bev, xy, task
        self.aug, self.flip = augment, flip
        self.bxy = np.stack(w2b(xy[..., 0], xy[..., 1]), -1).astype(np.float32)
    def __len__(self):
        return len(self.bev)
    def __getitem__(self, i):
        img = np.asarray(self.bev[i], np.float32) / 255.0
        t = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))
        t = torch.cat([t, torch.full((1, BEV_H, BEV_W), float(self.task[i]))], 0)
        bxy = torch.from_numpy(self.bxy[i]).clone()
        xy = torch.from_numpy(self.xy[i].astype(np.float32)).clone()
        if self.aug:
            t[:3] = (t[:3] * (0.85 + 0.30 * torch.rand(1))).clamp(0, 1)      # brightness
            t[:3] = (t[:3] + 0.03 * (torch.rand(1) - .5)).clamp(0, 1)        # offset
            t[:3] = (t[:3] * (0.96 + 0.08 * torch.rand(3, 1, 1))).clamp(0, 1)  # mild per-channel gain
        if self.flip and torch.rand(1).item() < 0.5:
            t = torch.flip(t, [2])
            bxy[..., 0] = BEV_W - bxy[..., 0]
            xy[..., 0] = (X0 + X1) - xy[..., 0]
        return t, bxy, xy

def make_loader(bev, xy, task, bs, train=False, augment=False, flip=False, workers=2):
    ds = BEVDataset(bev, xy, task, augment=augment, flip=flip)
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=train, num_workers=workers,
                                       pin_memory=True, drop_last=train,
                                       persistent_workers=workers > 0)

In [ ]:
def run_epoch(model, dl, opt, scaler, train=True, l1w=8.0, regw=0.15):
    model.train(train)
    tot, n = 0.0, 0
    for x, bxy, xy in dl:
        x = x.to(dev, non_blocking=True); bxy = bxy.to(dev); xy = xy.to(dev)
        with torch.autocast("cuda", enabled=(dev.type == "cuda")):
            logits, reg = model(x)
            hm_t = heatmap_targets(bxy, logits.shape[2], logits.shape[3])
            lh = focal_heatmap_loss(logits, hm_t)
            sa = soft_argmax(logits)
            wx, wy = b2w(sa[..., 0], sa[..., 1])
            l1 = (torch.stack([wx, wy], -1) - xy).abs().mean()
            wrx, wry = b2w(reg[..., 0], reg[..., 1])
            lreg = (torch.stack([wrx, wry], -1) - xy).abs().mean()
            loss = lh + l1w * l1 + regw * lreg
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        tot += float(loss.detach()) * len(x); n += len(x)
    return tot / max(n, 1)

@torch.no_grad()
def predict_xy(model, dl, blend=0.0):
    model.eval()
    out = []
    for x, bxy, xy in dl:
        x = x.to(dev, non_blocking=True)
        with torch.autocast("cuda", enabled=(dev.type == "cuda")):
            logits, reg = model(x)
            sa = soft_argmax(logits)
            p = (1 - blend) * sa + blend * reg
            wx, wy = b2w(p[..., 0], p[..., 1])
        out.append(torch.stack([wx, wy], -1).float().cpu().numpy())
    return np.concatenate(out)

def rmse(a, b):
    return float(np.sqrt(((a - b) ** 2).mean()))

## 7 - Training

90 / 10 split for model selection, OneCycle LR, AMP, `DataParallel` across the two T4s.
`EPOCHS` trades runtime for accuracy.

In [ ]:
EPOCHS       = 35
BATCH        = 48
LR           = 3e-3
WIDTH        = 32
VAL_FRAC     = 0.10
FLIP_AUG     = False      # mirror-world augmentation (doubles the data)

xy_tr = A_tr[:, :, :2]
task_tr = (train_df.task_type == "task2").astype(int).values
task_te = (test_df.task_type == "task2").astype(int).values

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(train_df))
nval = int(len(train_df) * VAL_FRAC)
val_idx, trn_idx = np.sort(perm[:nval]), np.sort(perm[nval:])
print("train/val:", len(trn_idx), len(val_idx))

dl_trn = make_loader(BEV_TR[trn_idx], xy_tr[trn_idx], task_tr[trn_idx], BATCH, True, True)
dl_val = make_loader(BEV_TR[val_idx], xy_tr[val_idx], task_tr[val_idx], BATCH, False, False, False, 0)

In [ ]:
seed_all()
model = Net(width=WIDTH).to(dev)
if NGPU > 1:
    model = nn.DataParallel(model)
opt = torch.optim.AdamW(model.parameters(), LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl_trn),
                                            pct_start=0.25, div_factor=10, final_div_factor=20)
scaler = torch.amp.GradScaler("cuda", enabled=(dev.type == "cuda"))

best = (1e9, None)
hist = {"ep": [], "tr": [], "val": []}
t0 = time.time()
for ep in range(EPOCHS):
    tr_loss = run_epoch(model, dl_trn, opt, scaler, train=True)
    sched.step()
    if ep % 3 == 2 or ep == EPOCHS - 1:
        p = predict_xy(model, dl_val)
        v = rmse(p, xy_tr[val_idx])
        hist["ep"].append(ep + 1); hist["tr"].append(tr_loss); hist["val"].append(v)
        flag = ""
        if v < best[0]:
            best = (v, {k: t.detach().clone() for k, t in model.state_dict().items()})
            flag = "  *best"
        print(f"ep {ep+1:3d}/{EPOCHS}  loss {tr_loss:7.3f}  val RMSE {v:.4f} m  "
              f"({time.time()-t0:.0f}s){flag}", flush=True)
print("best val RMSE %.4f m" % best[0])

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(hist["ep"], hist["val"], "o-", label="val RMSE (m)")
plt.xlabel("epoch"); plt.ylabel("RMSE"); plt.legend(); plt.grid(alpha=.3)
plt.title("best validation RMSE = %.4f m" % best[0]); plt.tight_layout(); plt.show()

## 8 - Predict the test split with the best validation checkpoint

Validation RMSE starts to degrade once the small network memorises the training scenes, so the
checkpoint with the lowest held-out error is used for inference instead of a long full-data
fine-tune.

In [ ]:
model_f = Net(width=WIDTH).to(dev)
if NGPU > 1:
    model_f = nn.DataParallel(model_f)
if best[1] is not None:
    model_f.load_state_dict(best[1])
    print("using best validation checkpoint, val RMSE %.4f m" % best[0])
else:
    print("WARNING: no checkpoint captured")

In [ ]:
dl_te = make_loader(BEV_TE, np.zeros((len(test_df), NSTEP, 2), np.float32), task_te,
                    BATCH, False, False, False, 0)
xy_te = predict_xy(model_f, dl_te)
print("test predictions:", xy_te.shape,
      "| x range %.3f..%.3f" % (xy_te[..., 0].min(), xy_te[..., 0].max()),
      "| y range %.3f..%.3f" % (xy_te[..., 1].min(), xy_te[..., 1].max()))

## 9 - Build `submission.csv`

In [ ]:
def build_submission(df, xy, zg_lookup, path="submission.csv"):
    rows = []
    for i, (_, r) in enumerate(df.iterrows()):
        out = {}
        for s in range(NSTEP):
            x, y = float(xy[i, s, 0]), float(xy[i, s, 1])
            zb, gb = zg_lookup[(r.task_type, s)]
            out[f"action_{s}"] = f"{x:.4f},{y:.4f},{zb:.4f},{gb:.4f}"
        rows.append({"image_id": r.image_id, **out})
    sub = pd.DataFrame(rows).set_index("image_id").loc[df.image_id].reset_index()
    sub.to_csv(path, index=False)
    return sub

sub = build_submission(test_df, xy_te, ZG, "submission.csv")
if os.path.isdir("/kaggle/working"):
    sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape); sub.head()

In [ ]:
sample = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
assert list(sub.columns) == list(sample.columns), (list(sub.columns), list(sample.columns))
assert (sub.image_id.values == sample.image_id.values).all()
print("format check passed  |  rows:", len(sub))
print(sub.iloc[0].to_dict())

### Notes / next steps
* The 12 `(z, grip)` values are copied, so the whole score is decided by the 12 `(x, y)` values.
* The BEV removes camera variation; the network only has to learn *which* object each step refers to
  (the ordering follows the rendered colour/shape layout, which the model picks up on its own).
* Further gains if more compute is available: `FLIP_AUG=True`, a 5-fold ensemble, and a stride-1
  heat-map head.